# Combine Static and Photic EEG Datasets

## Objective
The purpose of this notebook is to load the preprocessed derivative data from both the resting-state (`ds004504`) and photic-stimulation (`ds006036`) datasets, combine them into a single unified dataset, and save the combined epochs to disk.

This ensures we have a single, anti-leakage combined dataset ready for joint model training.


In [2]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Add src to path to use utility functions
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT / "src"))

from utl.bids import load_bids_dataset
from utl.eeg import load_and_format_data

# Define data paths (targeting local preprocessed derivative data)
STATIC_DATA_PATH = PROJECT_ROOT / "data" / "ds004504"
PHOTIC_DATA_PATH = PROJECT_ROOT / "data" / "ds006036"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "joint_dataset"

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
print("--- Loading Static Dataset Metadata (ds004504) ---")
df_static, (static_cn, static_ad, _) = load_bids_dataset(str(STATIC_DATA_PATH))
print(f"Static Subjects: {len(static_cn)} CN, {len(static_ad)} AD")

print("\n--- Loading Photic Dataset Metadata (ds006036) ---")
df_photic, (photic_cn, photic_ad, _) = load_bids_dataset(str(PHOTIC_DATA_PATH))
print(f"Photic Subjects: {len(photic_cn)} CN, {len(photic_ad)} AD")

# Tag the data before merging
df_static['condition'] = 'static'
df_photic['condition'] = 'photic'

# Combine metadata
df_joint = pd.concat([df_static, df_photic], ignore_index=True)

# Build a unified list of unique subjects
all_cn = list(set(static_cn) | set(photic_cn))
all_ad = list(set(static_ad) | set(photic_ad))
all_subjects = all_cn + all_ad

print(f"\n--- Combined Dataset Stats ---")
print(f"Total Unique Subjects: {len(all_subjects)} ({len(all_cn)} CN, {len(all_ad)} AD)")
print(f"Total Recordings: {len(df_joint[df_joint['Group'].isin(['C', 'A'])])}")

# Save the combined metadata
df_joint.to_csv(OUTPUT_DIR / "joint_metadata.csv", index=False)
print(f"Saved metadata to {OUTPUT_DIR / 'joint_metadata.csv'}")


--- Loading Static Dataset Metadata (ds004504) ---
Static Subjects: 29 CN, 36 AD

--- Loading Photic Dataset Metadata (ds006036) ---
Photic Subjects: 29 CN, 36 AD

--- Combined Dataset Stats ---
Total Unique Subjects: 65 (29 CN, 36 AD)
Total Recordings: 130
Saved metadata to /Users/buraa/Repo/detached-eeg/data/processed/joint_dataset/joint_metadata.csv


## Load & Format EEG Trials
We will now use the metadata to locate all the actual `.set` or `.edf` files on the disk, extract the 5-second trials, and concatenate them into a massive unified array.


In [4]:
print(f"Loading and formatting data for all {len(all_subjects)} subjects...")
print("This will take a few minutes as it processes gigabytes of EEG data...")

# Extract trials using the utility function
X_all, y_all, s_indices = load_and_format_data(df_joint, all_subjects, max_trials_per_subject=None)

print(f"\n✅ Data Loading Complete!")
print(f"Total trials loaded: {X_all.shape[0]} across {len(all_subjects)} subjects.")
print(f"Features (X) shape: {X_all.shape} -> (Trials, Channels, Timepoints)")
print(f"Labels (y) shape: {y_all.shape}")
print(f"Subject index (s) shape: {s_indices.shape}")


Loading and formatting data for all 65 subjects...
This will take a few minutes as it processes gigabytes of EEG data...


/Users/buraa/Repo/detached-eeg/src/utl/eeg.py:57: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw(row['eeg_file'], preload=True, verbose=False)
/Users/buraa/Repo/detached-eeg/src/utl/eeg.py:57: RuntimeWarning: Limited 3 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw(row['eeg_file'], preload=True, verbose=False)
/Users/buraa/Repo/detached-eeg/src/utl/eeg.py:57: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw(row['eeg_file'], preload=True, verbose=False)
/Users/buraa/Repo/detached-eeg/src/utl/eeg.py:57: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw(row['eeg_file'], preload=True, verbose=False)
/Users/buraa/Repo/detached-eeg/src/utl/eeg.py:57: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw(row['eeg_file'], preload=T


✅ Data Loading Complete!
Total trials loaded: 14649 across 65 subjects.
Features (X) shape: (14649, 19, 640) -> (Trials, Channels, Timepoints)
Labels (y) shape: (14649,)
Subject index (s) shape: (14649,)


## Save Combined Dataset
To make future training scripts faster, we will save these extracted numpy arrays to a compressed `.npz` file. 


In [5]:
output_file = OUTPUT_DIR / "joint_trials.npz"
print(f"Saving combined arrays to {output_file}...")

# Save as compressed numpy archive
np.savez_compressed(
    output_file,
    X=X_all,
    y=y_all,
    subject_indices=s_indices,
    subject_list=np.array(all_subjects) # Save the list to map indices back to names
)

print("Done! You can now load this dataset in any other script using:")
print("data = np.load('data/processed/joint_dataset/joint_trials.npz')")
print("X, y, s = data['X'], data['y'], data['subject_indices']")


Saving combined arrays to /Users/buraa/Repo/detached-eeg/data/processed/joint_dataset/joint_trials.npz...
Done! You can now load this dataset in any other script using:
data = np.load('data/processed/joint_dataset/joint_trials.npz')
X, y, s = data['X'], data['y'], data['subject_indices']
